<img src="https://github.com/radevicstefan-afk/DSB-Unit-03/blob/main/assets/ga-logo.png?raw=1" style="float: left; margin: 20px; height: 55px">

# Train Test Split and Cross Validation - Solution

---

### About
An introduction to creating generalizable models using methods such as train, test, split, and describing the bias variance tradeoff.


### Learning Objectives
- Describe the principles of Generalizability.
- Compare Bias and Variance and describe the tradeoff between when selecting a model.
- Define and apply the Train Test Split method using `scikit-learn`.
- Explore and apply the k-Folds Cross Validation method in `scikit-learn`.


### Notebook Guide

- Creating Models That Are Generalizable
- Part I: The Train-Test Split
- Now You Try
- Part II: Cross Validation
- Now You Try
- Conclusion and Takeaways

In [23]:
# import libraries we will use

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

# Creating Models That Are Generalizable
---

## Overfitting

We run into a problem when powerful models can perfectly fit the data on which they are trained. These models are **low bias** and **high variance**. Because we only know how the model fits the data we have, rather than the underlying population of interest, we don't have a good picture of how it work with new (aka: "outside") data or "in the real world".
<br /><br />
![overfit](https://github.com/radevicstefan-afk/DSB-Unit-03/blob/main/assets/overfit.png?raw=1)

**Solution:** Create a procedure that estimates how well a model is likely to perform on out-of-sample data and use that to choose between models. Overall, our model will likely have poorer performance on our data sample, but will be more generalizable or work better overall.


## Making a Prediction From a Sample

We often assume that our data set contains every sample for a given problem. In other words, the whole population.  In cases like the Titanic data set, that is true. Although it's convenient, in the real world all you typically have is a **sample** of data. This may sound like a contentious statement, but the point of machine learning is to generalize from a sample of data to the underlying population. If you already have data for an entire population, then you have no need for machine learning: you can apply statistics directly and get optimal answers!

So how can we get our models to generalize well?

**Methods:**   
1. **Train-test-split**:
     - Is useful if cross-validation is not practical (e.g. it takes too long to train)
     - It is also useful for computing a quick confusion matrix (for classification models)
     - You could also use this as a final step after the model is finalized (often called evaluating the model against a validation set)
1. **Cross-validation** is the gold standard for estimating accuracy and comparing accuracy across models for many model types.
1. **Three-way split** combines cross-validation and the train-test-split. It takes an initial split to be used as a final validation set, then uses cross-validation on the rest.

# Part I: The Train-Test Split
---

![](https://github.com/radevicstefan-afk/DSB-Unit-03/blob/main/assets/train_test.png?raw=1)

1. Split the data set into two pieces: a **training set** and a **testing set**.
2. Train the model on the **training set**.
3. Test the model on the **testing set** and evaluate how well we did.

Often a good rule-of-thumb is 70% training / 30% test, but this can vary based on the size of your data set. For example, with a small data set you would need to use as much training data as possible (in return, your test accuracy will be more variable).

What does this accomplish?

- Models can be trained and tested on **different data** (we treat testing data like out-of-sample data).
- Response values are known for the testing set and thus **predictions can be evaluated**.

In [24]:
# read in the bikeshare data

data = pd.read_csv("https://raw.githubusercontent.com/radevicstefan-afk/DSB-Unit-03/refs/heads/main/data/bikeshare.csv")
data.head(10)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,0,1,1
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,0,1,1
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2,0,2
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,1,2,3
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,1,7,8
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,8,6,14


In [25]:
# import the train_test_split module from scikit-learn, and split the data

from sklearn.model_selection import train_test_split

train, test = train_test_split(data, test_size=0.3)

In [26]:
# inspect the shape of the data that has been split

print("Full data set shape:", data.shape)
print("Train data set shape:", train.shape, "% of Total:", round(len(train)/len(data)*100, 2))
print("Test data set shape:", test.shape, "% of Total:", round(len(test)/len(data)*100, 2))
# this gives you a 70:30 split

Full data set shape: (10886, 12)
Train data set shape: (7620, 12) % of Total: 70.0
Test data set shape: (3266, 12) % of Total: 30.0


In [27]:
data.columns

Index(['datetime', 'season', 'holiday', 'workingday', 'weather', 'temp',
       'atemp', 'humidity', 'windspeed', 'casual', 'registered', 'count'],
      dtype='object')

In [28]:
# define a feature matrix for a model

#X are the columns you are using as input variables to predict Y
X = data[['weather', 'temp', 'humidity', 'windspeed']]

#Y is variable you're trying to predict
y = data['count']

In [29]:
# we often read in features so that we can do the split and have X and y features set up for us

X_train, X_test, y_train, y_test = train_test_split(X, y)

In [30]:
# inspect the outputs

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (8164, 4)
y_train shape: (8164,)
X_test shape: (2722, 4)
y_test shape: (2722,)


In [ ]:
# sometimes you need to (approximately) keep the proportion of y values through the training and test sets
# to do this, use the stratify parameter

#  Now You Try
Use the `scikit-learn` [docs on **Train Test Split**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) to help you **create a train test split with a 70:30 split on X and y**.

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=12345)

In [32]:
print("Full shape:", data.shape)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

Full shape: (10886, 12)
X_train shape: (7620, 4)
y_train shape: (7620,)
X_test shape: (3266, 4)
y_test shape: (3266,)


### Fit a Linear Regression model on the training data

In [33]:
# instanitate your Linear Regression
model = LinearRegression()

In [34]:
# fit your model on the features (X) and the target (y)
model.fit(X_train, y_train) #set to train

LinearRegression()

### Check the metrics on the training and the test data

In [35]:
# Regression Metrics - training

y_pred = model.predict(X_train)
print("MSE", round(mean_squared_error(y_train, y_pred), 3))
print("Adj R Sq", round(model.score(X_train, y_train), 3))

MSE 25085.141
Adj R Sq 0.24


In [36]:
# Regression Metrics - testing

y_pred = model.predict(X_test)
print("MSE", round(mean_squared_error(y_test, y_pred), 3))
print("Adj R Sq", round(model.score(X_test, y_test), 3))

MSE 24441.844
Adj R Sq 0.244


The model performs better on the test data than the training data (this can happen).

---

To the slides!

---

# Part II: Cross Validation
---

Train/test split provides us with helpful tool, but it's a shame that we are tossing out a large chunk of our data for testing purposes.

**How can we use the maximum amount of our data points while still ensuring model integrity?**

1. Split our data into a number of different pieces (folds).
2. Train using `k-1` folds for training and a different fold for testing.
3. Average our model against EACH of those iterations.
4. Choose our model and TEST it against the final fold.
5. Average all test accuracies to get the estimated out-of-sample accuracy.

Although this may sound complicated, we are just training the model on k separate train-test-splits, then taking the average of the resulting test accuracies!

![](https://github.com/radevicstefan-afk/DSB-Unit-03/blob/main/assets/cross_validation_diagram.png?raw=1)

### Run K-Folds with 3, then 5 splits and observe the output.
What do you notice?

[K-Fold Docs](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)   

In [37]:
# import
from sklearn.model_selection import KFold

In [38]:
kf = KFold(n_splits=3, shuffle=True, random_state= 1318)

In [41]:
# manually run a 3-fold cross-validation

Rsq_values = []
n = 0

print("CROSS VALIDATION each fold")
for train_index, test_index in kf.split(X, y):
    lm = LinearRegression().fit(X.iloc[train_index], y.iloc[train_index])

    Rsq_values.append(lm.score(X.iloc[test_index], y.iloc[test_index]))

    n += 1

    print(f'Model {n}')
    print(f'R Square: {Rsq_values[n-1]}')


print("SUMMARY OF CROSS VALIDATION")
print(f'Mean of R Square for all folds: {np.mean(Rsq_values)}')

CROSS VALIDATION each fold
Model 1
R Square: 0.2271409269641781
Model 2
R Square: 0.2539555762941994
Model 3
R Square: 0.24048921199256812
SUMMARY OF CROSS VALIDATION
Mean of R Square for all folds: 0.2405285717503152


### Edit the above code to have 5 folds and observe the output.
What do you notice?

In [42]:
kf = KFold(n_splits=5, shuffle=True, random_state= 1318)

In [43]:
# manually run a 5-fold cross-validation

Rsq_values = []
n = 0

print("~~~~ CROSS VALIDATION each fold ~~~~")
for train_index, test_index in kf.split(X, y):
    lm = LinearRegression().fit(X.iloc[train_index], y.iloc[train_index])

    Rsq_values.append(lm.score(X.iloc[test_index], y.iloc[test_index]))

    n += 1

    print(f'Model {n}')
    print(f'R Square: {Rsq_values[n-1]}')


print("~~~~ SUMMARY OF CROSS VALIDATION ~~~~")
print(f'Mean of R Square for all folds: {np.mean(Rsq_values)}')

~~~~ CROSS VALIDATION each fold ~~~~
Model 1
R Square: 0.23743649945408052
Model 2
R Square: 0.22980407307963335
Model 3
R Square: 0.2596919967419832
Model 4
R Square: 0.23604958921087194
Model 5
R Square: 0.24109638974956593
~~~~ SUMMARY OF CROSS VALIDATION ~~~~
Mean of R Square for all folds: 0.24081570964722698


The scores are approximately the same.

### Cross-validation score
`cross_val_score` makes this a lot easier to code: [docs](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)

In [44]:
# import
from sklearn.model_selection import cross_val_score

# note the results will vary each run since we take a different
# subset of the data each time (since shuffle=True)

kf = KFold(n_splits=5, shuffle=True, random_state= 1318)
lm = LinearRegression()

In [45]:
# run cross_val_score and print scores

scores= cross_val_score(lm, X, y, cv=kf)
print(f'CV R Square {scores}, \nAverage R Square {scores.mean()}')

CV R Square [0.2374365  0.22980407 0.259692   0.23604959 0.24109639], 
Average R Square 0.24081570964722698


Don't want to to use **R-square** for your score?  Using this for a different model type? Check out your scoring options [here](https://scikit-learn.org/stable/modules/model_evaluation.html)

#  Now You Try
1. Run a Linear Regression using cross validation with k folds creating 10 splits.
1. Run a Linear Regression using cross validation with `cross_val_score` with k folds creating 5 splits.
1. Your choice: edit the parameters of `kfolds` or `cross_val_score`.

#### Run a Linear Regression  using cross validation with k folds creating 10 splits

In [46]:
kf = KFold(n_splits=10, shuffle=True, random_state= 1318)
scores= cross_val_score(lm, X, y, cv=kf)
print(f'CV R Square {scores}, \nAverage R Square {scores.mean()}')

CV R Square [0.25052809 0.22537691 0.21082714 0.24717064 0.26606707 0.2530089
 0.25682478 0.21563492 0.25267414 0.2290749 ], 
Average R Square 0.240718749826876


#### Run a Linear Regression  using cross validation with `cross_val_score` with k folds creating 5 splits

In [47]:
kf = KFold(n_splits=10, shuffle=True, random_state= 1318)
scores= cross_val_score(lm, X, y, cv=kf)
print(f'CV R Square {scores}, \nAverage R Square {scores.mean()}')

CV R Square [0.25052809 0.22537691 0.21082714 0.24717064 0.26606707 0.2530089
 0.25682478 0.21563492 0.25267414 0.2290749 ], 
Average R Square 0.240718749826876


# Recap

1. Define overfitting, and underfitting for models
1. Describe 2 methods to make your models more generalizable
1. Apply train, test, split and K-Folds cross validations to bikeshare data